## Setup

In [1]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import json
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Attention
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from tqdm.auto import tqdm
from time import perf_counter
import warnings
warnings.filterwarnings('ignore')

# Agregar src al path
BASE_DIR = Path.cwd().parents[1]
sys.path.append(str(BASE_DIR / 'src'))

from config import PROCESSED_DIR, logger

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

TARGET_COMMODITIES = ['Corn', 'Soybeans', 'Wheat']

print(f"✓ Base directory: {BASE_DIR}")
print(f"✓ Target commodities: {', '.join(TARGET_COMMODITIES)}")
print(f"✓ TensorFlow: {tf.__version__}")
print(f"✓ GPU disponible: {tf.config.list_physical_devices('GPU')}")

✓ Base directory: c:\Users\trico\OneDrive\UBA\Taller de Programacion\TPFinal
✓ Target commodities: Corn, Soybeans, Wheat
✓ TensorFlow: 2.18.0
✓ GPU disponible: []


## 1. Cargar Dataset

In [2]:
# Cargar dataset completo
input_file = PROCESSED_DIR / 'features_selected_modeling.csv'
df = pd.read_csv(input_file, parse_dates=['date'])

# Cargar precios spot
base_file = PROCESSED_DIR / 'commodities_base_consolidated.csv'
df_base = pd.read_csv(base_file, parse_dates=['date'])
spot_cols = {c: f'{c}_spot' for c in TARGET_COMMODITIES}
df_spots = df_base[['date'] + TARGET_COMMODITIES].rename(columns=spot_cols)
df = df.merge(df_spots, on='date', how='left')

# Definir features y targets
target_cols = [f'{c}_target_t7' for c in TARGET_COMMODITIES]
spot_cols_list = [f'{c}_spot' for c in TARGET_COMMODITIES]
feature_cols = [c for c in df.columns if c not in ['date'] + target_cols + spot_cols_list]

print(f"✓ Dataset: {df.shape}")
print(f"  Rango: {df['date'].min()} → {df['date'].max()}")
print(f"  Features: {len(feature_cols)}")
print(f"  Targets: {len(target_cols)}")

✓ Dataset: (6724, 51)
  Rango: 2000-01-03 00:00:00 → 2025-10-30 00:00:00
  Features: 44
  Targets: 3


## 2. Función: Construir LSTM Multivariado (del Notebook 3.7)

In [3]:
def build_multivariate_lstm(n_features, seq_len=30, lstm_units=64, dropout=0.3):
    """
    Construye LSTM multivariado con Bidirectional y Dropout.
    Arquitectura del notebook 3.7 (simplificada sin Attention para velocidad).
    """
    model = Sequential([
        Bidirectional(LSTM(lstm_units, return_sequences=True), 
                     input_shape=(seq_len, n_features)),
        Dropout(dropout),
        Bidirectional(LSTM(lstm_units // 2)),
        Dropout(dropout),
        Dense(32, activation='relu'),
        Dropout(dropout / 2),
        Dense(1)
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    return model

print("✓ Función build_multivariate_lstm definida")

✓ Función build_multivariate_lstm definida


## 3. Función: Crear Secuencias para LSTM

In [4]:
def create_sequences(X, y, seq_len=30):
    """
    Convierte features tabulares en secuencias para LSTM.
    X: (n_samples, n_features) → (n_samples - seq_len, seq_len, n_features)
    """
    X_seq, y_seq = [], []
    
    for i in range(seq_len, len(X)):
        X_seq.append(X[i-seq_len:i])
        y_seq.append(y[i])
    
    return np.array(X_seq), np.array(y_seq)

print("✓ Función create_sequences definida")

✓ Función create_sequences definida


## 4. Walk-Forward Validation con LSTM

In [5]:
def walk_forward_lstm(
    df, feature_cols, target_col, spot_col,
    initial_train_size=1500,
    step_size=60,
    seq_len=30,
    lstm_units=64,
    epochs=50,
    batch_size=32,
    verbose=True
):
    """
    Walk-forward validation con LSTM multivariado.
    
    DIFERENCIAS vs sklearn (notebook 3.9):
    - Requiere crear secuencias (seq_len=30)
    - Scaling separado por fold
    - Early stopping para evitar sobreajuste
    - Mucho más lento (re-entrenar red neuronal cada fold)
    """
    results = []
    
    # Preparar datos
    df_sorted = df.sort_values('date').reset_index(drop=True)
    
    X = df_sorted[feature_cols].copy()
    y = df_sorted[target_col].copy()
    spot = df_sorted[spot_col].copy()
    dates = df_sorted['date'].copy()
    
    # Limpieza
    X = X.replace([np.inf, -np.inf], np.nan).fillna(X.median().fillna(0))
    
    total_obs = len(df_sorted)
    current_idx = initial_train_size
    fold = 0
    
    if verbose:
        print(f"\nIniciando walk-forward LSTM:")
        print(f"  Initial train size: {initial_train_size}")
        print(f"  Step size: {step_size}")
        print(f"  Sequence length: {seq_len}")
        print(f"  Epochs por fold: {epochs}")
        print(f"  Folds estimados: ~{(total_obs - initial_train_size) // step_size}")
    
    with tqdm(total=(total_obs - initial_train_size), desc="LSTM Walk-Forward") as pbar:
        while current_idx + step_size <= total_obs:
            fold += 1
            
            # Definir ventanas
            train_start = 0
            train_end = current_idx
            test_start = current_idx
            test_end = min(current_idx + step_size, total_obs)
            
            X_train = X.iloc[train_start:train_end].values
            y_train = y.iloc[train_start:train_end].values
            X_test = X.iloc[test_start:test_end].values
            y_test = y.iloc[test_start:test_end].values
            spot_test = spot.iloc[test_start:test_end].values
            dates_test = dates.iloc[test_start:test_end]
            
            # Scaling
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train)
            X_test_scaled = scaler.transform(X_test)
            
            # Crear secuencias
            X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train, seq_len)
            X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test, seq_len)
            
            # Ajustar spot_test para secuencias
            spot_test_seq = spot_test[seq_len:]
            
            # Verificar que hay suficientes datos
            if len(X_train_seq) < 100 or len(X_test_seq) < 10:
                pbar.update(step_size)
                continue
            
            # Construir y entrenar modelo
            model = build_multivariate_lstm(
                n_features=X_train_seq.shape[2],
                seq_len=seq_len,
                lstm_units=lstm_units,
                dropout=0.3
            )
            
            early_stop = EarlyStopping(
                monitor='val_loss',
                patience=10,
                restore_best_weights=True,
                verbose=0
            )
            
            # Entrenar (silenciosamente)
            model.fit(
                X_train_seq, y_train_seq,
                validation_split=0.2,
                epochs=epochs,
                batch_size=batch_size,
                callbacks=[early_stop],
                verbose=0
            )
            
            # Predecir
            y_pred = model.predict(X_test_seq, verbose=0).flatten()
            
            # Métricas
            rmse = np.sqrt(mean_squared_error(y_test_seq, y_pred))
            mae = mean_absolute_error(y_test_seq, y_pred)
            r2 = r2_score(y_test_seq, y_pred)
            
            # Directional accuracy
            y_test_direction = np.sign(y_test_seq - spot_test_seq)
            y_pred_direction = np.sign(y_pred - spot_test_seq)
            dir_acc = (y_test_direction == y_pred_direction).mean()
            
            results.append({
                'fold': fold,
                'train_size': train_end - train_start,
                'test_size': len(y_test_seq),
                'train_start_date': dates.iloc[train_start],
                'train_end_date': dates.iloc[train_end - 1],
                'test_start_date': dates_test.iloc[seq_len],
                'test_end_date': dates_test.iloc[-1],
                'rmse': rmse,
                'mae': mae,
                'r2': r2,
                'dir_acc': dir_acc
            })
            
            # Limpiar memoria
            del model
            tf.keras.backend.clear_session()
            
            current_idx += step_size
            pbar.update(step_size)
            pbar.set_postfix({'Fold': fold, 'RMSE': f'{rmse:.2f}', 'R²': f'{r2:.3f}', 'DA': f'{dir_acc:.2%}'})
    
    results_df = pd.DataFrame(results)
    
    if verbose:
        print(f"\n✓ Walk-forward LSTM completado: {fold} folds")
        print(f"  RMSE promedio: {results_df['rmse'].mean():.4f} ± {results_df['rmse'].std():.4f}")
        print(f"  R² promedio: {results_df['r2'].mean():.4f} ± {results_df['r2'].std():.4f}")
        print(f"  Dir Acc promedio: {results_df['dir_acc'].mean():.2%} ± {results_df['dir_acc'].std():.2%}")
    
    return results_df

print("✓ Función walk_forward_lstm definida")

✓ Función walk_forward_lstm definida


## 5. Ejecutar Walk-Forward: LSTM Multivariado

**ADVERTENCIA:** Esto puede tardar 30-60 minutos por commodity (87 folds × 50 epochs × 3 commodities).

In [ ]:
lstm_wf_results = {}

print(f"\n{'='*80}")
print(f"WALK-FORWARD VALIDATION: LSTM MULTIVARIADO")
print(f"{'='*80}")
print(f"⚠️  ADVERTENCIA: Proceso intensivo (~30-60 min por commodity)")
print(f"    - Entrenar red neuronal 87 veces por commodity")
print(f"    - Total: ~261 entrenamientos LSTM")

for commodity in TARGET_COMMODITIES:
    target_col = f'{commodity}_target_t7'
    spot_col = f'{commodity}_spot'
    
    print(f"\n{'─'*80}")
    print(f"--- {commodity} ---")
    print(f"{'─'*80}")
    
    start_time = perf_counter()
    
    results_df = walk_forward_lstm(
        df, feature_cols, target_col, spot_col,
        initial_train_size=1500,
        step_size=60,
        seq_len=30,
        lstm_units=64,
        epochs=50,
        batch_size=32,
        verbose=True
    )
    
    elapsed = perf_counter() - start_time
    
    lstm_wf_results[commodity] = results_df
    
    print(f"\n✓ {commodity} completado en {elapsed/60:.1f} minutos")

print(f"\n{'='*80}")
print(f"✓ LSTM walk-forward completado para {len(TARGET_COMMODITIES)} commodities")
print(f"{'='*80}")


WALK-FORWARD VALIDATION: LSTM MULTIVARIADO
⚠️  ADVERTENCIA: Proceso intensivo (~30-60 min por commodity)
    - Entrenar red neuronal 87 veces por commodity
    - Total: ~261 entrenamientos LSTM

────────────────────────────────────────────────────────────────────────────────
--- Corn ---
────────────────────────────────────────────────────────────────────────────────

Iniciando walk-forward LSTM:
  Initial train size: 1500
  Step size: 60
  Sequence length: 30
  Epochs por fold: 50
  Folds estimados: ~87


LSTM Walk-Forward:   0%|          | 0/5224 [00:00<?, ?it/s]

2025-12-06 16:25:17 - tensorflow - WARNING - From f:\miniconda\envs\ds\Lib\site-packages\keras\src\backend\common\global_state.py:82: The name tf.reset_default_graph is deprecated. Please use tf.compat.v1.reset_default_graph instead.



2025-12-06 16:26:47 - tensorflow - WARNING - 5 out of the last 5 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x0000017DE4B1B4C0> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.


2025-12-06 16:27:04 - tensorflow - WARNING - 6 out of the last 6 calls to <function TensorFlowTrainer.make_predict_function.<locals>.one_step_on_data_distributed at 0x0000017DF10BFC40> triggered tf.function retracing. Tracing is expensive and the excessive number of tracings could be due to (1) creating @tf.function repeatedly in a loop, (2) passing tensors with different shapes, (3) passing Python objects instead of tensors. For (1), please define your @tf.function outside of the loop. For (2), @tf.function has reduce_retracing=True option that can avoid unnecessary retracing. For (3), please refer to https://www.tensorflow.org/guide/function#controlling_retracing and https://www.tensorflow.org/api_docs/python/tf/function for  more details.



✓ Walk-forward LSTM completado: 87 folds
  RMSE promedio: 110.7447 ± 89.3650
  R² promedio: -168.3437 ± 519.3614
  Dir Acc promedio: 49.81% ± 17.28%

✓ Corn completado en 58.8 minutos

────────────────────────────────────────────────────────────────────────────────
--- Soybeans ---
────────────────────────────────────────────────────────────────────────────────

Iniciando walk-forward LSTM:
  Initial train size: 1500
  Step size: 60
  Sequence length: 30
  Epochs por fold: 50
  Folds estimados: ~87


LSTM Walk-Forward:   0%|          | 0/5224 [00:00<?, ?it/s]

## 6. Comparación: LSTM vs Lasso vs Random Forest

In [ ]:
# Cargar resultados de Lasso y RF del notebook 3.9
wf_file = PROCESSED_DIR / 'walk_forward_validation_results.json'
with open(wf_file, 'r') as f:
    wf_data = json.load(f)

comparison_wf = []

for commodity in TARGET_COMMODITIES:
    # Lasso (del notebook 3.9)
    lasso_df = pd.DataFrame(wf_data[f'lasso_{commodity.lower()}_folds'])
    comparison_wf.append({
        'Commodity': commodity,
        'Model': 'Lasso',
        'Avg RMSE': lasso_df['rmse'].mean(),
        'Std RMSE': lasso_df['rmse'].std(),
        'Avg R²': lasso_df['r2'].mean(),
        'Std R²': lasso_df['r2'].std(),
        'Avg Dir Acc': lasso_df['dir_acc'].mean(),
        'Num Folds': len(lasso_df)
    })
    
    # Random Forest (del notebook 3.9)
    rf_df = pd.DataFrame(wf_data[f'rf_{commodity.lower()}_folds'])
    comparison_wf.append({
        'Commodity': commodity,
        'Model': 'Random Forest',
        'Avg RMSE': rf_df['rmse'].mean(),
        'Std RMSE': rf_df['rmse'].std(),
        'Avg R²': rf_df['r2'].mean(),
        'Std R²': rf_df['r2'].std(),
        'Avg Dir Acc': rf_df['dir_acc'].mean(),
        'Num Folds': len(rf_df)
    })
    
    # LSTM (nuevo)
    lstm_df = lstm_wf_results[commodity]
    comparison_wf.append({
        'Commodity': commodity,
        'Model': 'LSTM Multivariado',
        'Avg RMSE': lstm_df['rmse'].mean(),
        'Std RMSE': lstm_df['rmse'].std(),
        'Avg R²': lstm_df['r2'].mean(),
        'Std R²': lstm_df['r2'].std(),
        'Avg Dir Acc': lstm_df['dir_acc'].mean(),
        'Num Folds': len(lstm_df)
    })

comparison_wf_df = pd.DataFrame(comparison_wf)

print(f"\n{'='*80}")
print(f"COMPARACIÓN WALK-FORWARD: LASSO vs RF vs LSTM")
print(f"{'='*80}\n")

for commodity in TARGET_COMMODITIES:
    print(f"\n--- {commodity} ---")
    commodity_results = comparison_wf_df[comparison_wf_df['Commodity'] == commodity]
    display(commodity_results[['Model', 'Avg RMSE', 'Std RMSE', 'Avg R²', 'Std R²', 'Avg Dir Acc']])
    
    # Identificar mejor modelo
    best_idx = commodity_results['Avg R²'].idxmax()
    best_model = commodity_results.loc[best_idx, 'Model']
    best_r2 = commodity_results.loc[best_idx, 'Avg R²']
    print(f"\n✓ Mejor modelo: {best_model} (R² promedio: {best_r2:.4f})")

print(f"\n{'='*80}")

## 7. Visualización: R² por Modelo y Commodity

In [ ]:
# Bar plot: Avg R² comparison
fig, ax = plt.subplots(figsize=(12, 6))

commodities = comparison_wf_df['Commodity'].unique()
models = comparison_wf_df['Model'].unique()
x = np.arange(len(commodities))
width = 0.25

for idx, model in enumerate(models):
    model_data = comparison_wf_df[comparison_wf_df['Model'] == model]
    r2_values = [model_data[model_data['Commodity'] == c]['Avg R²'].values[0] for c in commodities]
    ax.bar(x + idx * width, r2_values, width, label=model, alpha=0.8)

ax.set_xlabel('Commodity', fontsize=12)
ax.set_ylabel('Promedio R² (Walk-Forward)', fontsize=12)
ax.set_title('Walk-Forward R²: LSTM vs Lasso vs Random Forest', fontsize=14, fontweight='bold')
ax.set_xticks(x + width)
ax.set_xticklabels(commodities)
ax.legend(loc='best')
ax.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(BASE_DIR / 'reports' / 'figures' / 'walk_forward_comparison_lstm.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Gráfico guardado: walk_forward_comparison_lstm.png")

## 8. Guardar Resultados

In [ ]:
# Guardar resultados LSTM walk-forward
lstm_wf_summary = {
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'metodologia': 'Walk-Forward Validation (LSTM Multivariado)',
    'initial_train_size': 1500,
    'step_size': 60,
    'seq_len': 30,
    'lstm_units': 64,
    'epochs': 50,
    'commodities': TARGET_COMMODITIES,
    'comparison': comparison_wf_df.to_dict(orient='records')
}

# Agregar folds detallados
for commodity in TARGET_COMMODITIES:
    lstm_wf_summary[f'lstm_{commodity.lower()}_folds'] = lstm_wf_results[commodity].to_dict(orient='records')

results_file = PROCESSED_DIR / 'walk_forward_lstm_results.json'
with open(results_file, 'w') as f:
    json.dump(lstm_wf_summary, f, indent=2, default=str)

print(f"✓ Resultados guardados: {results_file}")

# Guardar DataFrames individuales
for commodity in TARGET_COMMODITIES:
    lstm_wf_results[commodity].to_csv(
        PROCESSED_DIR / f'wf_lstm_{commodity.lower()}.csv', index=False
    )

print(f"✓ DataFrames guardados: 3 archivos CSV (LSTM × 3 commodities)")

---

## Conclusiones Walk-Forward LSTM

**Hallazgos esperados:**

1. **Si LSTM mantiene R² positivo:** Demuestra superioridad sobre Lasso/RF
2. **Si LSTM también colapsa (R² negativo):** Problema es estructural de los datos, no del modelo
3. **Directional Accuracy:** Métrica crítica - LSTM debe superar 55% para justificar complejidad

**Próximos pasos según resultados:**
- **Si LSTM funciona:** Priorizar LSTM/VMD-LSTM para producción
- **Si LSTM falla también:** Considerar features adicionales (sentiment, COT, supply/demand)
- **Trade-off:** LSTM es 100x más lento que Lasso - debe justificar complejidad con DA >55%